In [1]:
import numpy as np
from iminuit import Minuit
from iminuit.cost import ExtendedBinnedNLL
from scipy.stats import expon, norm
import plotly.graph_objects as go


def exp( x , A , tau ):
    return expon.cdf(x , A , tau) 

def gauss( x , N , mu , sigma):
    return N*norm.cdf(x , mu , sigma)

def exp_gauss_cdf(N_exp , A , tau):
    def fixed_exp( x , N , mu , sigma):
        return exp( x , N_exp , A , tau) + gauss( x , N , mu , sigma)
    return fixed_exp
    
def gauss_exp_cdf(N_g , mu , sigma):
    def fixed_gauss( x , N , A , tau):
        return exp( x , N , A , tau) + gauss( x , N_g , mu , sigma)
    return fixed_gauss

def exp_fit(cdf, A , tau, count , edges):
    
    
    cost = ExtendedBinnedNLL(count, edges, cdf)
    n = Minuit(cost, A = A , tau = tau )
    # n.fixed['N'] = True
    n.migrad(ncall = 1000000)
    n.hesse()
    return n

def gauss_fit( cdf , N , mu , sigma , count , edges):
    cost = ExtendedBinnedNLL(count, edges, cdf)
    n = Minuit(cost, N = N , mu = mu , sigma = sigma )
    n.migrad(ncall = 1000000)
    n.hesse()
    return n

In [20]:
data_1 = np.genfromtxt("Data/timestamp/14_1_26_17_39.csv", delimiter=',')
# print(len(data_1))
data_1 = data_1[(data_1 > 1e-9) & (data_1 < 1e-4)]
print(np.std(data_1))
n_bins = 100

count, edges = np.histogram(data_1, bins=n_bins , density=False)

fig = go.Figure()

fig.add_trace(
    go.Bar(x=edges[:-1], y=count, name='Histogram', width=np.diff(edges))
)

fig.update_layout(
    xaxis_title='Time (s)',
    yaxis_title='Counts',
    bargap=0
)

fig.show()

4.566189194786616e-06
